In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [9]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"),"store")    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)



+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Warszawa|765      |270876.64|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Gdańsk  |766      |302579.07|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Wrocław |798      |327127.76|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Kraków  |821      |341327.83|
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Kraków  |1169     |483309.86|
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Warszawa|1117     |451638.0 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Gdańsk  |1174     |488279.58|
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Wrocław |1201     |473002.77|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|Wrocław |557      |202608.68|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|Gdańsk  |558      |230407.7 |
|{2026-04-12 10:00:00, 2026-04-12 11:0

# PRACA DOMOWA

### Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.\n

In [28]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"),"store")    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+--------+---------+-----------+
|window                                    |store   |liczba_tx|srednia_PLN|
+------------------------------------------+--------+---------+-----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Warszawa|765      |354.09     |
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Gdańsk  |766      |395.01     |
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Wrocław |798      |409.93     |
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|Kraków  |821      |415.75     |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Kraków  |1169     |413.44     |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Warszawa|1117     |404.33     |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Gdańsk  |1174     |415.91     |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|Wrocław |1201     |393.84     |
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|Wrocław |557      |363.75     |
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|Gdańsk  |558      |412.92     |
|{2026-04-12

In [31]:
(
    (hourly
    .filter(col("store") == "Gdańsk")
    .orderBy(col("srednia_PLN").asc())
     .select(
        col("store").alias("od"),
        "liczba_tx",
        "srednia_PLN",
    )
    .show(truncate=False)
)
)

+------+---------+-----------+
|od    |liczba_tx|srednia_PLN|
+------+---------+-----------+
|Gdańsk|766      |395.01     |
|Gdańsk|558      |412.92     |
|Gdańsk|1174     |415.91     |
+------+---------+-----------+



In [32]:
from pyspark.sql.functions import min
(hourly
    .filter(col("store") == "Gdańsk")
    .agg(min("srednia_PLN"))
    .show()
    )


+----------------+
|min(srednia_PLN)|
+----------------+
|          395.01|
+----------------+



### Policz ile transakcji per kategoria było w oknie 09:00–09:30.\n

In [34]:
from pyspark.sql.functions import window


hourly = (
    df.groupBy(window("timestamp", "30 minutes"),"category")    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+-----------+---------+---------+
|window                                    |category   |liczba_tx|suma_PLN |
+------------------------------------------+-----------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|elektronika|269      |148911.79|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|odzież     |302      |103945.14|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|książki    |268      |80235.23 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|żywność    |273      |78067.65 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|książki    |527      |196101.87|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|odzież     |506      |175335.63|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|żywność    |486      |152929.56|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|elektronika|519      |306384.43|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |349852.93|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |204888.51|

In [39]:
from pyspark.sql.functions import col

(hourly
    .filter(col("window.start") == "2026-04-12 09:00:00")
    .show(truncate=False)
)

+------------------------------------------+-----------+---------+---------+
|window                                    |category   |liczba_tx|suma_PLN |
+------------------------------------------+-----------+---------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |349852.93|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |204888.51|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |191895.44|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |175645.23|
+------------------------------------------+-----------+---------+---------+




### Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

In [52]:
from pyspark.sql.functions import window


hourly = (
    df.groupBy(window("timestamp", "15 minutes"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468      |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644      |
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899      |
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139     |
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171     |
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156     |
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100     |
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858      |
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582      |
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|443      |
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|306      |
+------------------------------------------+---------+



In [57]:
from pyspark.sql.functions import col

(hourly
    .orderBy(col("liczba_tx").desc())
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .show(1, truncate=False)
)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+
only showing top 1 row

